# Baseline vs LLM agent: ticket triage

Flow of this notebook:
1. **pandas**: load and explore the tickets
2. **scikit-learn**: train a cheap TF-IDF + logistic regression baseline
3. **matplotlib**: confusion matrix of the baseline
4. **LLM agent** (your LangGraph project): run it on the same test tickets
5. **Compare** accuracy, human-review rate and confidence calibration

The sample data is tiny (12 training, 8 test tickets), so treat the numbers as a demo of the method, not as results. Swap in your own anonymized tickets in `data/` and rerun.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 140)

RUN_LLM = True   # set False to skip live LLM calls and reuse evals/results.csv if it exists


## 1. Load and explore (pandas)

`resolved_tickets.csv` is the history we learn from. `golden_tickets.csv` is the human-labelled test set the models never see during training.

In [ ]:
train = pd.read_csv("data/resolved_tickets.csv")
test = pd.read_csv("data/golden_tickets.csv")
print("train:", train.shape, "| test:", test.shape)
train.head()


In [ ]:
print("Tickets per category (train):")
print(train["category"].value_counts(), "\n")
print("Category vs team (train):")
display(pd.crosstab(train["category"], train["team"]))
print("Test set: priority by true team")
display(pd.crosstab(test["true_team"], test["true_priority"]))


## 2. Baseline classifier (scikit-learn)

TF-IDF turns text into weighted word counts, and logistic regression learns which words point to which team. It is fast, free, and runs offline. `predict_proba` gives a real model probability, which we use as its confidence.

In [ ]:
def make_baseline():
    return make_pipeline(
        TfidfVectorizer(stop_words="english", ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000),
    )

base_team = make_baseline().fit(train["text"], train["team"])
base_cat = make_baseline().fit(train["text"], train["category"])

baseline = test[["ticket_id", "text", "true_category", "true_team"]].copy()
baseline["pred_category"] = base_cat.predict(test["text"])
baseline["pred_team"] = base_team.predict(test["text"])
baseline["confidence"] = base_team.predict_proba(test["text"]).max(axis=1)
baseline["category_ok"] = baseline["pred_category"] == baseline["true_category"]
baseline["team_ok"] = baseline["pred_team"] == baseline["true_team"]

print(f"Baseline category accuracy: {baseline.category_ok.mean():.0%}")
print(f"Baseline team accuracy    : {baseline.team_ok.mean():.0%}")
baseline[["ticket_id", "true_team", "pred_team", "confidence", "team_ok"]]


In [ ]:
print(classification_report(baseline["true_team"], baseline["pred_team"], zero_division=0))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    baseline["true_team"], baseline["pred_team"], xticks_rotation=45, ax=ax, colorbar=False
)
ax.set_title("Baseline: true vs predicted team")
plt.tight_layout()
plt.show()


## 3. The LLM agent (your LangGraph project)

If an API key is set in `.env` and `RUN_LLM = True`, this runs the real agent on the test tickets in batch mode (risky tickets go to the human queue instead of pausing). Otherwise it reuses `evals/results.csv` from an earlier `python -m evals.run_eval` run. If neither exists, the comparison cells are skipped.

In [ ]:
import time
from dotenv import load_dotenv

load_dotenv()
has_key = bool(os.getenv("OPENAI_API_KEY"))  # Gemini key from .env
RESULTS = Path("evals/results.csv")
agent = None

if RUN_LLM and has_key:
    from app.runner import run_ticket

    delay = float(os.getenv("EVAL_DELAY", "2"))
    rows = []
    for _, r in test.iterrows():
        out = run_ticket(str(r["ticket_id"]), r["text"], mode="batch")
        rows.append({
            "ticket_id": r["ticket_id"], "route": out["route"],
            "pred_category": out["category"], "pred_priority": out["priority"],
            "pred_team": out["team"], "confidence": out["confidence"],
            "reasons": "; ".join(out["reasons"]),
        })
        time.sleep(delay)
    agent = pd.DataFrame(rows)
    print("Ran the live agent on", len(agent), "tickets")
elif RESULTS.exists():
    agent = pd.read_csv(RESULTS)
    print("Loaded saved agent results from", RESULTS)
else:
    print("No API key and no saved results: skipping the agent comparison.")

if agent is not None:
    agent = agent.merge(test[["ticket_id", "true_category", "true_priority", "true_team"]],
                        on="ticket_id", suffixes=("_old", ""))
    agent = agent[[c for c in agent.columns if not c.endswith("_old")]]
    agent["category_ok"] = agent["pred_category"] == agent["true_category"]
    agent["priority_ok"] = agent["pred_priority"] == agent["true_priority"]
    agent["team_ok"] = agent["pred_team"] == agent["true_team"]
    display(agent[["ticket_id", "route", "pred_team", "true_team", "confidence", "team_ok"]])


## 4. Side-by-side comparison

In [ ]:
if agent is None:
    print("Agent results not available yet.")
else:
    auto = agent[agent["route"] == "auto_assigned"]
    summary = pd.DataFrame({
        "Baseline (TF-IDF + LogReg)": {
            "category accuracy": baseline.category_ok.mean(),
            "team accuracy": baseline.team_ok.mean(),
            "sent to humans": 0.0,
            "accuracy on auto-assigned": baseline.team_ok.mean(),
        },
        "LLM agent": {
            "category accuracy": agent.category_ok.mean(),
            "team accuracy": agent.team_ok.mean(),
            "sent to humans": (agent["route"] != "auto_assigned").mean(),
            "accuracy on auto-assigned": auto.team_ok.mean() if len(auto) else np.nan,
        },
    })
    display(summary.style.format("{:.0%}"))
    summary.T[["category accuracy", "team accuracy"]].plot(kind="bar", ylim=(0, 1), rot=0, figsize=(6, 4))
    plt.title("Accuracy: baseline vs agent")
    plt.show()


In [ ]:
if agent is not None:
    cmp = (
        baseline[["ticket_id", "text", "true_team", "pred_team"]]
        .rename(columns={"pred_team": "baseline_team"})
        .merge(
            agent[["ticket_id", "pred_team", "confidence", "route"]]
            .rename(columns={"pred_team": "agent_team", "confidence": "agent_conf"}),
            on="ticket_id",
        )
    )
    cmp["baseline_ok"] = cmp["baseline_team"] == cmp["true_team"]
    cmp["agent_ok"] = cmp["agent_team"] == cmp["true_team"]
    print("Tickets where the two disagree or either is wrong:")
    display(cmp[~(cmp.baseline_ok & cmp.agent_ok)][
        ["ticket_id", "text", "true_team", "baseline_team", "agent_team", "agent_conf", "route"]
    ])


## 5. Is the confidence score trustworthy? (calibration)

If confidence means anything, accuracy should rise with it. Bands with few tickets are noisy, so you need a few hundred labelled tickets before trusting this table. Use it to choose `CONFIDENCE_THRESHOLD` in `app/config.py`.

In [ ]:
bands = [0, 0.5, 0.75, 0.9, 1.0]

def calibration(df, label):
    d = df.copy()
    d["band"] = pd.cut(d["confidence"], bands, include_lowest=True)
    out = d.groupby("band", observed=True).agg(tickets=("team_ok", "size"), accuracy=("team_ok", "mean"))
    print(label)
    display(out.style.format({"accuracy": "{:.0%}"}))

calibration(baseline, "Baseline (model probability)")
if agent is not None:
    calibration(agent, "LLM agent (self-reported confidence)")


## How to read this

- **If the agent does not clearly beat the baseline** on your real labelled data, the extra cost, latency and complexity of an LLM may not be justified.
- **Accuracy on auto-assigned** is the number to watch before enabling auto-assignment. It should be high, and it only counts tickets the agent was allowed to route without a human.
- **Calibration**: if the top confidence band is not clearly more accurate than the lower ones, do not trust the confidence score for gating until you improve it (neighbour agreement, self-consistency, or a combined score).
- A common production design is to use both: the baseline as a cheap second opinion, and send tickets to a human when the two disagree.

Next steps: replace the CSVs with 100 to 300 of your own anonymized tickets (keeping train and test separate), rerun, then tune the prompt and threshold.
